# Day-Ahead 24-Hour Forecasting at 15-Minute Granularity (TFT) — Anti-Overfitting + Walk-Forward CV

This notebook keeps the full preprocessing pipeline from the forecasting notebook and adds:
- stronger regularization to reduce overfitting
- walk-forward cross-validation for robust model selection

## 1) Install dependencies (only if needed)
Run once if imports fail, then restart kernel.

In [2]:
%pip install -q pytorch-forecasting lightning holidays plotly

Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import warnings
from datetime import datetime
from pathlib import Path

import holidays
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

warnings.filterwarnings("ignore")
pl.seed_everything(42, workers=True)
torch.set_float32_matmul_precision("medium")

Seed set to 42


In [4]:
# Paths + full preprocessing (same as forecasting notebook)
PROJECT_ROOT = Path.cwd().parent.parent.parent.parent
PRICE_PATH = PROJECT_ROOT / "data" / "price_data.csv"
WEATHER_PATH = PROJECT_ROOT / "data" / "historical_hourly_2023_2025.csv"
WEATHER_DAILY = PROJECT_ROOT / "data" / "historical_daily_2023_2025.csv"
OUTPUT_PATH = PROJECT_ROOT / "Prediction" / "tft_day_ahead_24h_15min.csv"

price_df = pd.read_csv(PRICE_PATH)
ts_col = "timestamp" if "timestamp" in price_df.columns else "Timestamp"
price_df[ts_col] = pd.to_datetime(price_df[ts_col], utc=True, errors="coerce").dt.tz_convert(None)
price_df = price_df.rename(columns={ts_col: "Timestamp"})
price_df = price_df.sort_values("Timestamp").drop_duplicates("Timestamp")

base_cols = [
    c for c in [
        "Timestamp", "price", "demand_itsdo", "demand_indo", "demand_inddem",
        "demand_forecast", "wind_generation", "solar_generation", "margin_daily_forecast"
    ] if c in price_df.columns
]
price_df = price_df[base_cols]

full_index = pd.date_range(price_df["Timestamp"].min(), price_df["Timestamp"].max(), freq="15min")
price_df = price_df.set_index("Timestamp").reindex(full_index).rename_axis("Timestamp").reset_index()

weather_df = pd.read_csv(WEATHER_PATH)
w_ts = "Timestamp" if "Timestamp" in weather_df.columns else "timestamp_utc"
weather_df[w_ts] = pd.to_datetime(weather_df[w_ts], utc=True, errors="coerce").dt.tz_convert(None)
weather_df = weather_df.rename(columns={w_ts: "WeatherTimestamp"}).sort_values("WeatherTimestamp")

hourly_keep = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m",
    "precipitation", "rain", "snowfall",
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "shortwave_radiation", "direct_radiation",
    "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m",
    "surface_pressure", "weather_code"
]
hourly_keep = [c for c in hourly_keep if c in weather_df.columns]
weather_df = weather_df[["WeatherTimestamp"] + hourly_keep]

weather_df = weather_df.set_index("WeatherTimestamp")
weather_15min_index = pd.date_range(price_df["Timestamp"].min(), price_df["Timestamp"].max(), freq="15min")
weather_15min = weather_df.reindex(weather_15min_index)

numeric_weather_cols = [c for c in weather_15min.columns if c != "weather_code"]
if numeric_weather_cols:
    weather_15min[numeric_weather_cols] = weather_15min[numeric_weather_cols].interpolate(method="time", limit_direction="both")
if "weather_code" in weather_15min.columns:
    weather_15min["weather_code"] = weather_15min["weather_code"].ffill().bfill()

weather_15min = weather_15min.reset_index().rename(columns={"index": "Timestamp"})
df = price_df.merge(weather_15min, on="Timestamp", how="left")

daily_df = pd.read_csv(WEATHER_DAILY)
d_ts = "date_utc" if "date_utc" in daily_df.columns else "Timestamp"
daily_df[d_ts] = pd.to_datetime(daily_df[d_ts], utc=True, errors="coerce").dt.tz_convert(None)
daily_df = daily_df.rename(columns={d_ts: "Timestamp"}).sort_values("Timestamp")

if {"sunrise", "daylight_duration"}.issubset(daily_df.columns):
    sunrise = pd.to_datetime(daily_df["sunrise"], errors="coerce").dt.tz_localize(None)
    day_length_hours = daily_df["daylight_duration"] / 3600.0
    safe_day_length = day_length_hours.replace(0, np.nan)
    solar_noon_dt = sunrise + pd.to_timedelta(day_length_hours / 2, unit="h")
    hours = (daily_df["Timestamp"] - solar_noon_dt) / np.timedelta64(1, "h")
    daily_df["Solar_intensity"] = np.maximum(0, np.cos(hours * np.pi / safe_day_length)).fillna(0.0)

excluded_daily_cols = {"Timestamp", "sunrise", "sunset", "daylight_duration", "sunshine_duration"}
daily_numeric = [c for c in daily_df.columns if c not in excluded_daily_cols and pd.api.types.is_numeric_dtype(daily_df[c])]
daily_features = daily_df[["Timestamp"] + daily_numeric].drop_duplicates("Timestamp")
daily_features = daily_features.set_index("Timestamp")
daily_15min = daily_features.reindex(weather_15min_index).ffill().bfill().reset_index().rename(columns={"index": "Timestamp"})
df = df.merge(daily_15min, on="Timestamp", how="left")

for col in df.columns:
    if col == "Timestamp":
        continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].interpolate(limit_direction="both").ffill().bfill()

df = df[df["Timestamp"] > "2025-01-01"]
df = df.dropna(subset=["Timestamp", "price"]).sort_values("Timestamp").reset_index(drop=True)

print(f"Rows: {len(df):,}")
print(f"Range: {df['Timestamp'].min()} -> {df['Timestamp'].max()}")
display(df.head(3))

Rows: 40,571
Range: 2025-01-01 00:15:00 -> 2026-02-27 14:45:00


,Timestamp,price,demand_itsdo,demand_indo,demand_inddem,demand_forecast,wind_generation,solar_generation,margin_daily_forecast,temperature_2m,...,weather_code_x,weather_code_y,shortwave_radiation_sum,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,temperature_2m_max,temperature_2m_min,Solar_intensity
0,2025-01-01 00:15:00,9.07,26483.0,20949.0,-173.0,20900.0,20557.559,0.0,19370.0,6.4875,...,61.0,61.0,1.57,10.400001,10.400001,0.0,17.0,6.6,-2.85,0.492076
1,2025-01-01 00:30:00,23.51,26474.0,21129.0,-2338.0,20900.0,20108.433,0.0,19370.0,6.5250,...,61.0,61.0,1.57,10.400001,10.400001,0.0,17.0,6.6,-2.85,0.492076
2,2025-01-01 00:45:00,23.51,26474.0,21129.0,-2338.0,20900.0,20108.433,0.0,19370.0,6.5625,...,61.0,61.0,1.57,10.400001,10.400001,0.0,17.0,6.6,-2.85,0.492076


In [5]:
# Patch known bad price segment
if "price" not in df.columns:
    raise KeyError("Expected 'price' column in df")

if "price_raw" not in df.columns:
    df["price_raw"] = df["price"]
if "is_price_patched" not in df.columns:
    df["is_price_patched"] = False

target_year = int(df["Timestamp"].dt.year.mode().iloc[0])
target_day = pd.Timestamp(f"{target_year}-01-08").date()
start_time = pd.to_datetime("11:15").time()
end_time = pd.to_datetime("21:00").time()

window_mask = ((df["Timestamp"].dt.date == target_day) & (df["Timestamp"].dt.time >= start_time) & (df["Timestamp"].dt.time <= end_time))
bad_mask = window_mask & (df["price"] > 200)

ref_start = pd.Timestamp(f"{target_year}-01-01").date()
ref_end = pd.Timestamp(f"{target_year}-01-15").date()
reference_mask = ((df["Timestamp"].dt.date >= ref_start) & (df["Timestamp"].dt.date <= ref_end) & (df["Timestamp"].dt.date != target_day) & (df["price"] <= 200))

slot_median = df.loc[reference_mask].groupby(df.loc[reference_mask, "Timestamp"].dt.time)["price"].median()
replacement_values = df.loc[bad_mask, "Timestamp"].dt.time.map(slot_median)
fallback_median = float(df.loc[reference_mask, "price"].median()) if reference_mask.any() else float(df["price"].median())
replacement_values = replacement_values.fillna(fallback_median)

df.loc[bad_mask, "price"] = replacement_values.values
df.loc[bad_mask, "is_price_patched"] = True
print(f"Patched rows: {int(bad_mask.sum())}")

Patched rows: 36


In [6]:
# Feature engineering + split setup
uk_holidays = holidays.CountryHoliday("UK")
df["hour"] = df["Timestamp"].dt.hour.astype(str)
df["quarter_hour"] = (df["Timestamp"].dt.minute // 15).astype(str)
df["quarter_of_day"] = ((df["Timestamp"].dt.hour * 60 + df["Timestamp"].dt.minute) // 15).astype(str)
df["day_of_week"] = df["Timestamp"].dt.dayofweek.astype(str)
df["is_weekend"] = (df["Timestamp"].dt.dayofweek >= 5).astype(str)
df["is_holiday"] = df["Timestamp"].dt.date.isin(uk_holidays).astype(str)
df["is_working_day"] = ((df["is_weekend"] == "False") & (df["is_holiday"] == "False")).astype(str)

minute_of_day = df["Timestamp"].dt.hour * 60 + df["Timestamp"].dt.minute
df["tod_sin"] = np.sin(2 * np.pi * minute_of_day / 1440.0)
df["tod_cos"] = np.cos(2 * np.pi * minute_of_day / 1440.0)
df["dow_sin"] = np.sin(2 * np.pi * df["Timestamp"].dt.dayofweek / 7.0)
df["dow_cos"] = np.cos(2 * np.pi * df["Timestamp"].dt.dayofweek / 7.0)

df["series_id"] = "GB"
df["time_idx"] = np.arange(len(df), dtype=np.int64)

MAX_PREDICTION_LENGTH = 96
MAX_ENCODER_LENGTH = 288

known_weather = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m",
    "precipitation", "rain", "snowfall",
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "shortwave_radiation", "direct_radiation",
    "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m",
    "surface_pressure", "Solar_intensity"
]
price_known_reals = ["demand_itsdo","demand_indo","demand_inddem","demand_forecast","wind_generation","solar_generation","margin_daily_forecast"]
daily_known_reals = [c for c in daily_numeric if c in df.columns and c != "price"] if "daily_numeric" in globals() else []
known_reals = [c for c in known_weather if c in df.columns] + daily_known_reals + ["tod_sin", "tod_cos", "dow_sin", "dow_cos"] + price_known_reals
known_reals = list(dict.fromkeys(known_reals))

known_categoricals = [c for c in ["hour", "quarter_hour", "quarter_of_day", "day_of_week", "is_weekend", "is_holiday", "is_working_day"] if c in df.columns]
unknown_reals = ["price"]

VAL_STEPS = 7 * 24 * 4
TEST_STEPS = 7 * 24 * 4
train_cutoff = df.time_idx.max() - (VAL_STEPS + TEST_STEPS + MAX_PREDICTION_LENGTH)
val_start = train_cutoff + 1
test_start = df.time_idx.max() - (TEST_STEPS + MAX_PREDICTION_LENGTH) + 1

print(f"train_cutoff={train_cutoff}, val_start={val_start}, test_start={test_start}")

train_cutoff=39130, val_start=39131, test_start=39803


In [7]:
# Anti-overfitting hyperparameters
AF_MAX_EPOCHS = 60
AF_LR = 7e-4
AF_HIDDEN_SIZE = 16
AF_ATTN_HEADS = 2
AF_DROPOUT = 0.30
AF_HIDDEN_CONT = 8
AF_WEIGHT_DECAY = 1e-3
AF_EARLY_STOP_PATIENCE = 6
AF_REDUCE_ON_PLATEAU_PATIENCE = 3
BATCH_SIZE = 256
CV_FOLDS = 3

In [9]:
def make_datasets(train_cutoff_idx: int, val_start_idx: int, test_start_idx: int):
    training = TimeSeriesDataSet(
        df[df.time_idx <= train_cutoff_idx],
        time_idx="time_idx",
        target="price",
        group_ids=["series_id"],
        max_encoder_length=MAX_ENCODER_LENGTH,
        max_prediction_length=MAX_PREDICTION_LENGTH,
        time_varying_known_reals=known_reals,
        time_varying_known_categoricals=known_categoricals,
        time_varying_unknown_reals=unknown_reals,
        target_normalizer=GroupNormalizer(groups=["series_id"]),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
    )
    validation = TimeSeriesDataSet.from_dataset(training, df, min_prediction_idx=val_start_idx, stop_randomization=True)
    test_set = TimeSeriesDataSet.from_dataset(training, df, min_prediction_idx=test_start_idx, stop_randomization=True)

    train_loader = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
    val_loader = validation.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)
    test_loader = test_set.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)
    return training, validation, test_set, train_loader, val_loader, test_loader

def metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-6, None))) * 100

    pct_error = ((y_pred - y_true) / np.clip(np.abs(y_true), 1e-6, None)) * 100
    pct_error_mean = float(np.mean(pct_error))
    pct_error_median = float(np.median(pct_error))

    actual_pct_change = (np.diff(y_true) / np.clip(np.abs(y_true[:-1]), 1e-6, None)) * 100 if len(y_true) > 1 else np.array([])
    pred_pct_change = (np.diff(y_pred) / np.clip(np.abs(y_pred[:-1]), 1e-6, None)) * 100 if len(y_pred) > 1 else np.array([])
    pct_change_mae = float(np.mean(np.abs(pred_pct_change - actual_pct_change))) if len(actual_pct_change) else np.nan
    actual_pct_change_mean = float(np.mean(actual_pct_change)) if len(actual_pct_change) else np.nan
    pred_pct_change_mean = float(np.mean(pred_pct_change)) if len(pred_pct_change) else np.nan

    actual_up = actual_pct_change > 0
    actual_down = actual_pct_change < 0
    pred_up = pred_pct_change > 0
    pred_down = pred_pct_change < 0

    dir_match = np.sign(actual_pct_change) == np.sign(pred_pct_change)
    directional_accuracy = float(np.mean(dir_match)) if len(dir_match) else np.nan

    up_den = int(pred_up.sum())
    down_den = int(pred_down.sum())
    hit_rate_up = float((actual_up & pred_up).sum() / up_den) if up_den > 0 else np.nan
    hit_rate_down = float((actual_down & pred_down).sum() / down_den) if down_den > 0 else np.nan

    if len(actual_pct_change):
        spike_threshold = float(np.percentile(np.abs(actual_pct_change), 95))
        true_spike = np.abs(actual_pct_change) >= spike_threshold
        pred_spike = np.abs(pred_pct_change) >= spike_threshold

        tp = int((true_spike & pred_spike).sum())
        fp = int((~true_spike & pred_spike).sum())
        fn = int((true_spike & ~pred_spike).sum())

        spike_precision = float(tp / (tp + fp)) if (tp + fp) > 0 else np.nan
        spike_recall = float(tp / (tp + fn)) if (tp + fn) > 0 else np.nan
        spike_f1 = (
            float(2 * spike_precision * spike_recall / (spike_precision + spike_recall))
            if (spike_precision == spike_precision and spike_recall == spike_recall and (spike_precision + spike_recall) > 0)
            else np.nan
        )
    else:
        spike_threshold = np.nan
        spike_precision = np.nan
        spike_recall = np.nan
        spike_f1 = np.nan

    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "MAPE(%)": float(mape),
        "PctErrorMean(%ŷ-%y)": pct_error_mean,
        "PctErrorMedian(%ŷ-%y)": pct_error_median,
        "PctChangeMAE": pct_change_mae,
        "ActualPctChangeMean": actual_pct_change_mean,
        "PredPctChangeMean": pred_pct_change_mean,
        "DirectionalAccuracy": directional_accuracy,
        "HitRateUp": hit_rate_up,
        "HitRateDown": hit_rate_down,
        "SpikeThresholdAbsPctChg": spike_threshold,
        "SpikePrecision": spike_precision,
        "SpikeRecall": spike_recall,
        "SpikeF1": spike_f1,
    }

def predict_loader_median(model, loader):
    quantiles = [float(q) for q in model.loss.quantiles] if hasattr(model.loss, "quantiles") else []
    median_idx = int(np.argmin(np.abs(np.array(quantiles) - 0.5))) if quantiles else 0

    y_true_batches, y_pred_batches = [], []
    with torch.no_grad():
        for x, y in loader:
            out = model(x)
            pred_q = out["prediction"].detach().cpu().numpy()
            pred = pred_q[:, :, median_idx]
            true = y[0] if isinstance(y, (tuple, list)) else y
            true = true.detach().cpu().numpy()
            y_true_batches.append(true)
            y_pred_batches.append(pred)

    y_true_all = np.concatenate(y_true_batches, axis=0) if y_true_batches else np.empty((0, MAX_PREDICTION_LENGTH))
    y_pred_all = np.concatenate(y_pred_batches, axis=0) if y_pred_batches else np.empty((0, MAX_PREDICTION_LENGTH))
    return y_true_all, y_pred_all

def train_one_fold(fold_id: int, train_cutoff_idx: int, val_start_idx: int, test_start_idx: int):
    training, validation, test_set, train_loader, val_loader, test_loader = make_datasets(train_cutoff_idx, val_start_idx, test_start_idx)

    cv_dir = PROJECT_ROOT / "models" / "price_prediction" / "tft_day_ahead_15min_anti_overfit_cv" / f"fold_{fold_id}"
    cv_dir.mkdir(parents=True, exist_ok=True)

    early_stop = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=AF_EARLY_STOP_PATIENCE, mode="min")
    checkpoint_cb = ModelCheckpoint(dirpath=cv_dir, filename="tft-{epoch:02d}-{val_loss:.4f}", monitor="val_loss", mode="min", save_top_k=1, save_last=True)

    trainer = pl.Trainer(
        max_epochs=AF_MAX_EPOCHS,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        gradient_clip_val=0.1,
        callbacks=[early_stop, checkpoint_cb],
        logger=False,
        enable_checkpointing=True,
    )

    model = TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=AF_LR,
        hidden_size=AF_HIDDEN_SIZE,
        attention_head_size=AF_ATTN_HEADS,
        dropout=AF_DROPOUT,
        hidden_continuous_size=AF_HIDDEN_CONT,
        output_size=7,
        loss=QuantileLoss(),
        reduce_on_plateau_patience=AF_REDUCE_ON_PLATEAU_PATIENCE,
        weight_decay=AF_WEIGHT_DECAY
    )

    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
    best_model = TemporalFusionTransformer.load_from_checkpoint(checkpoint_cb.best_model_path).cpu().eval()

    val_true, val_pred = predict_loader_median(best_model, val_loader)
    test_true, test_pred = predict_loader_median(best_model, test_loader)

    val_metrics = metrics(val_true, val_pred)
    test_metrics = metrics(test_true, test_pred)

    row = {"fold": fold_id, "best_ckpt": checkpoint_cb.best_model_path}
    row.update({f"val_{k}": v for k, v in val_metrics.items()})
    row.update({f"test_{k}": v for k, v in test_metrics.items()})
    return row

In [8]:
# Walk-forward cross-validation
cv_rows = []
max_idx = int(df.time_idx.max())

for fold in range(CV_FOLDS):
    shift = (CV_FOLDS - 1 - fold) * TEST_STEPS
    fold_test_start = max_idx - (TEST_STEPS + MAX_PREDICTION_LENGTH) + 1 - shift
    fold_val_start = fold_test_start - VAL_STEPS
    fold_train_cutoff = fold_val_start - 1

    if fold_train_cutoff < (MAX_ENCODER_LENGTH + MAX_PREDICTION_LENGTH + 100):
        print(f"Skipping fold {fold}: not enough history")
        continue

    print(f"Fold {fold}: train_cutoff={fold_train_cutoff}, val_start={fold_val_start}, test_start={fold_test_start}")
    cv_rows.append(train_one_fold(fold, fold_train_cutoff, fold_val_start, fold_test_start))

cv_results_df = pd.DataFrame(cv_rows)
display(cv_results_df)

if len(cv_results_df):
    cv_summary_df = cv_results_df.drop(columns=["best_ckpt"], errors="ignore").mean(numeric_only=True).to_frame("mean")
    print("Walk-forward CV summary (mean across folds):")
    display(cv_summary_df)

Fold 0: train_cutoff=37786, val_start=37787, test_start=38459


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │  1.8 K │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    640 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  1.7 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 27.4 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 26.7 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │    808 │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    119 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 71.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 71.1 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 1322                                                                                        
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

Trainable params: 71.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 71.1 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 1322                                                                                        
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

,fold,best_ckpt,val_MAE,val_RMSE,val_MAPE(|y|>=10.0)(%),val_sMAPE(%),val_WAPE(%),val_MAPE_raw(%),val_DirectionalAccuracy,test_MAE,test_RMSE,test_MAPE(|y|>=10.0)(%),test_sMAPE(%),test_WAPE(%),test_MAPE_raw(%),test_DirectionalAccuracy
0,0,C:\Users\rkale\OneDrive\Documentos\VStudio\Git...,7.419565,10.783939,11.581922,10.945416,9.158830,21.442345,0.312834,7.999035,11.589155,13.203670,12.432425,10.358921,26.336408,0.310374
1,1,C:\Users\rkale\OneDrive\Documentos\VStudio\Git...,8.710364,12.135901,14.283946,13.047875,11.280107,28.531560,0.313214,9.536109,13.600698,17.175215,15.459931,13.032292,38.506946,0.322516
2,2,C:\Users\rkale\OneDrive\Documentos\VStudio\Git...,14.443761,17.202827,24.108903,21.265537,19.739215,41.591108,0.334149,13.915209,17.564585,28.828406,25.029239,20.925185,63.654631,0.345210


Walk-forward CV summary (mean across folds):


,mean
fold,1.000000
val_MAE,10.191230
val_RMSE,13.374223
val_MAPE(|y|>=10.0)(%),16.658257
val_sMAPE(%),15.086276
val_WAPE(%),13.392717
val_MAPE_raw(%),30.521671
val_DirectionalAccuracy,0.320066
test_MAE,10.483451
test_RMSE,14.251479


In [ ]:
# Final train/val/test on latest split (same as forecasting notebook, anti-overfit params)
training, validation, test_set, train_loader, val_loader, test_loader = make_datasets(train_cutoff, val_start, test_start)

checkpoint_dir = PROJECT_ROOT / "models" / "price_prediction" / "tft_day_ahead_15min_anti_overfit"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

early_stop = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=AF_EARLY_STOP_PATIENCE, mode="min")
checkpoint_cb = ModelCheckpoint(dirpath=checkpoint_dir, filename="tft-{epoch:02d}-{val_loss:.4f}", monitor="val_loss", mode="min", save_top_k=1, save_last=True)

trainer = pl.Trainer(
    max_epochs=AF_MAX_EPOCHS,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[early_stop, checkpoint_cb],
    enable_checkpointing=True,
    logger=False
)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=AF_LR,
    hidden_size=AF_HIDDEN_SIZE,
    attention_head_size=AF_ATTN_HEADS,
    dropout=AF_DROPOUT,
    hidden_continuous_size=AF_HIDDEN_CONT,
    output_size=7,
    loss=QuantileLoss(),
    reduce_on_plateau_patience=AF_REDUCE_ON_PLATEAU_PATIENCE,
    weight_decay=AF_WEIGHT_DECAY
)

print(f"Model params: {tft.size()/1e3:.1f}k")
trainer.fit(tft, train_dataloaders=train_loader, val_dataloaders=val_loader)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Model params: 71.1k


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │  1.8 K │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    640 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  1.7 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 27.4 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 26.7 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │    808 │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    119 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 71.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 71.1 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 1322                                                                                        
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Best checkpoint: C:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\models\price_prediction\tft_day_ahead_15min_anti_overfit\tft-epoch=04-val_loss=5.4091.ckpt
Last checkpoint: C:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\models\price_prediction\tft_day_ahead_15min_anti_overfit\last.ckpt


In [11]:
# Immediate Validation/Test metric reporting from checkpoint (can run standalone)
if "val_loader" not in globals() or "test_loader" not in globals():
    _, _, _, _, val_loader, test_loader = make_datasets(train_cutoff, val_start, test_start)

if "BEST_CHECKPOINT_PATH" not in globals():
    BEST_CHECKPOINT_PATH = ""
if "LAST_CHECKPOINT_PATH" not in globals():
    LAST_CHECKPOINT_PATH = ""

checkpoint_dir = PROJECT_ROOT / "models" / "price_prediction" / "tft_day_ahead_15min_anti_overfit"
if not BEST_CHECKPOINT_PATH and checkpoint_dir.exists():
    ckpts = sorted(checkpoint_dir.glob("*.ckpt"), key=lambda path: path.stat().st_mtime, reverse=True)
    if ckpts:
        BEST_CHECKPOINT_PATH = str(ckpts[0])
        LAST_CHECKPOINT_PATH = str(ckpts[0])

print("Best checkpoint:", BEST_CHECKPOINT_PATH)
print("Last checkpoint:", LAST_CHECKPOINT_PATH)

eval_ckpt = BEST_CHECKPOINT_PATH if BEST_CHECKPOINT_PATH else LAST_CHECKPOINT_PATH
if eval_ckpt and Path(eval_ckpt).exists():
    eval_tft = TemporalFusionTransformer.load_from_checkpoint(eval_ckpt).cpu().eval()
    val_true, val_pred = predict_loader_median(eval_tft, val_loader)
    test_true, test_pred = predict_loader_median(eval_tft, test_loader)

    print("Validation (96-step trajectories):")
    for k, v in metrics(val_true, val_pred).items():
        print(f"  {k}: {v:.4f}" if isinstance(v, (float, np.floating)) else f"  {k}: {v}")

    print("\nTest (96-step trajectories):")
    for k, v in metrics(test_true, test_pred).items():
        print(f"  {k}: {v:.4f}" if isinstance(v, (float, np.floating)) else f"  {k}: {v}")
else:
    print("No valid checkpoint found. Run the final training cell first, or set BEST_CHECKPOINT_PATH/LAST_CHECKPOINT_PATH.")

Best checkpoint: C:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\models\price_prediction\tft_day_ahead_15min_anti_overfit\tft-epoch=04-val_loss=5.4091.ckpt
Last checkpoint: C:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\models\price_prediction\tft_day_ahead_15min_anti_overfit\tft-epoch=04-val_loss=5.4091.ckpt
Validation (96-step trajectories):
  MAE: 9.8246
  RMSE: 13.7771
  MAPE(%): 38.8012
  PctErrorMean(%ŷ-%y): 34.1776
  PctErrorMedian(%ŷ-%y): 5.4967
  PctChangeMAE: 5.0098
  ActualPctChangeMean: 1.3656
  PredPctChangeMean: 0.0837
  DirectionalAccuracy: 0.3369
  HitRateUp: 0.2832
  HitRateDown: 0.3911
  SpikeThresholdAbsPctChg: 14.0685
  SpikePrecision: 0.7421
  SpikeRecall: 0.1513
  SpikeF1: 0.2513

Test (96-step trajectories):
  MAE: 12.8884
  RMSE: 17.3113
  MAPE(%): 68.7411
  PctErrorMean(%ŷ-%y): 64.0892
  PctErrorMedian(%ŷ-%y): 9.2645
  PctChangeMAE: 7.8713
  ActualPctChangeMean: 2.6641
  Pr

In [10]:
# Reload model and save metadata
CKPT_TO_LOAD = BEST_CHECKPOINT_PATH if BEST_CHECKPOINT_PATH else LAST_CHECKPOINT_PATH
if not CKPT_TO_LOAD:
    raise ValueError("No checkpoint path found. Train first or set CKPT_TO_LOAD manually.")

loaded_tft = TemporalFusionTransformer.load_from_checkpoint(CKPT_TO_LOAD)
loaded_tft.eval()

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
metadata_path = checkpoint_dir / f"run_metadata_{run_id}.json"
latest_metadata_path = checkpoint_dir / "latest_run_metadata.json"
run_metadata = {
    "run_id": run_id,
    "project_root": str(PROJECT_ROOT),
    "checkpoint_dir": str(checkpoint_dir),
    "best_checkpoint_path": BEST_CHECKPOINT_PATH,
    "last_checkpoint_path": LAST_CHECKPOINT_PATH,
    "model_class": "TemporalFusionTransformer",
    "max_prediction_length": int(MAX_PREDICTION_LENGTH),
    "max_encoder_length": int(MAX_ENCODER_LENGTH),
    "known_reals": known_reals,
    "known_categoricals": known_categoricals,
    "unknown_reals": unknown_reals,
    "train_cutoff": int(train_cutoff),
    "val_start": int(val_start),
    "test_start": int(test_start),
    "timestamp_min": str(df["Timestamp"].min()),
    "timestamp_max": str(df["Timestamp"].max()),
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(run_metadata, file, indent=2)
with open(latest_metadata_path, "w", encoding="utf-8") as file:
    json.dump(run_metadata, file, indent=2)

print("Loaded checkpoint:", CKPT_TO_LOAD)
print("Saved metadata:", metadata_path)

Loaded checkpoint: C:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\models\price_prediction\tft_day_ahead_15min_anti_overfit\tft-epoch=04-val_loss=5.4091.ckpt
Saved metadata: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\models\price_prediction\tft_day_ahead_15min_anti_overfit\run_metadata_20260302_210039.json


In [ ]:
# Detailed split evaluation + CSV/HTML exports (same as forecasting notebook)
model_for_eval = loaded_tft.cpu().eval()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

quantiles = [float(q) for q in model_for_eval.loss.quantiles] if hasattr(model_for_eval.loss, "quantiles") else []
median_idx = int(np.argmin(np.abs(np.array(quantiles) - 0.5))) if quantiles else 0

eval_batch_size = 16
train_eval_set = TimeSeriesDataSet.from_dataset(training, df[df.time_idx <= train_cutoff], stop_randomization=True)
train_eval_loader = train_eval_set.to_dataloader(train=False, batch_size=eval_batch_size, num_workers=0)
val_eval_loader = validation.to_dataloader(train=False, batch_size=eval_batch_size, num_workers=0)
test_eval_loader = test_set.to_dataloader(train=False, batch_size=eval_batch_size, num_workers=0)

time_lookup = df[["time_idx", "Timestamp"]].drop_duplicates("time_idx").set_index("time_idx")["Timestamp"]

def eval_with_time(loader):
    y_true_batches, y_pred_batches, time_batches = [], [], []
    with torch.no_grad():
        for x, y in loader:
            out = model_for_eval(x)
            pred_q = out["prediction"].detach().cpu().numpy()
            pred = pred_q[:, :, median_idx]
            true = y[0] if isinstance(y, (tuple, list)) else y
            true = true.detach().cpu().numpy()
            decoder_time_idx = x["decoder_time_idx"].detach().cpu().numpy()
            decoder_times = time_lookup.reindex(decoder_time_idx.reshape(-1)).to_numpy().reshape(decoder_time_idx.shape)
            y_true_batches.append(true)
            y_pred_batches.append(pred)
            time_batches.append(decoder_times)

    y_true_all = np.concatenate(y_true_batches, axis=0) if y_true_batches else np.empty((0, MAX_PREDICTION_LENGTH))
    y_pred_all = np.concatenate(y_pred_batches, axis=0) if y_pred_batches else np.empty((0, MAX_PREDICTION_LENGTH))
    y_time_all = np.concatenate(time_batches, axis=0) if time_batches else np.empty((0, MAX_PREDICTION_LENGTH), dtype="datetime64[ns]")
    return y_true_all, y_pred_all, y_time_all

train_true, train_pred, train_time = eval_with_time(train_eval_loader)
val_true, val_pred, val_time = eval_with_time(val_eval_loader)
test_true, test_pred, test_time = eval_with_time(test_eval_loader)

train_metrics = metrics(train_true, train_pred)
val_metrics = metrics(val_true, val_pred)
test_metrics = metrics(test_true, test_pred)

metrics_df = pd.DataFrame([
    {"Split": "Train", **train_metrics},
    {"Split": "Validation", **val_metrics},
    {"Split": "Test", **test_metrics},
]).set_index("Split")
display(metrics_df)

plots_dir = PROJECT_ROOT / "Prediction" / "plot_exports"
plots_dir.mkdir(parents=True, exist_ok=True)
metrics_path = plots_dir / "metrics_by_split_CV.csv"
metrics_df.to_csv(metrics_path)
print(f"Saved: {metrics_path}")

,MAE,RMSE,MAPE(|y|>=10.0)(%),sMAPE(%),WAPE(%),MAPE_raw(%),DirectionalAccuracy
Split,,,,,,,
Train,10.144955,14.879270,15.407908,20.826358,12.477816,3.326895e+06,0.328075
Validation,9.824570,13.777111,17.851864,16.032231,13.426510,3.880117e+01,0.336891
Test,12.888378,17.311308,27.093354,23.645289,19.381073,6.874113e+01,0.333447


Saved: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\Prediction\plot_exports\metrics_by_split_CV.csv


In [12]:
# Predict next 24h at 15-minute resolution
predict_ds = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)
predict_loader = predict_ds.to_dataloader(train=False, batch_size=1, num_workers=0)
raw_next, _, *_ = tft.predict(predict_loader, mode="raw", return_x=True)

next_96_pred = raw_next["prediction"][0, :, 3].detach().cpu().numpy()
last_ts = df["Timestamp"].max()
future_ts = pd.date_range(last_ts + pd.Timedelta(minutes=15), periods=96, freq="15min")

forecast_df = pd.DataFrame({"Timestamp": future_ts, "predicted_price": next_96_pred})
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
forecast_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
display(forecast_df.head())

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Saved: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\Prediction\tft_day_ahead_24h_15min.csv


,Timestamp,predicted_price
0,2026-02-27 15:00:00,83.812813
1,2026-02-27 15:15:00,81.579185
2,2026-02-27 15:30:00,85.806297
3,2026-02-27 15:45:00,86.809540
4,2026-02-27 16:00:00,86.843483


In [13]:
# Slider plots for train/validation/test windows
def build_window_slider_plot(split_name: str, y_true_split: np.ndarray, y_pred_split: np.ndarray, t_split: np.ndarray, out_path: Path):
    n_windows = len(y_true_split)
    if n_windows == 0:
        print(f"No windows available for {split_name}.")
        return

    x0 = pd.to_datetime(t_split[0])
    fig = go.Figure(data=[
        go.Scatter(x=x0, y=y_true_split[0], mode="lines", name="Real", line=dict(width=2)),
        go.Scatter(x=x0, y=y_pred_split[0], mode="lines", name="Predicted", line=dict(width=2, dash="dash")),
    ])

    frames = []
    for idx in range(n_windows):
        x_vals = pd.to_datetime(t_split[idx])
        ts_start = pd.to_datetime(t_split[idx][0])
        ts_end = pd.to_datetime(t_split[idx][-1])
        frames.append(go.Frame(name=str(idx), data=[go.Scatter(x=x_vals, y=y_true_split[idx]), go.Scatter(x=x_vals, y=y_pred_split[idx])], layout=go.Layout(title_text=f"{split_name}: Window #{idx} ({ts_start} to {ts_end})")))
    fig.frames = frames

    steps = [dict(method="animate", args=[[str(idx)], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}], label=str(idx)) for idx in range(n_windows)]

    first_start = pd.to_datetime(t_split[0][0])
    first_end = pd.to_datetime(t_split[0][-1])
    fig.update_layout(title=f"{split_name}: Window #0 ({first_start} to {first_end})", sliders=[dict(active=0, currentvalue={"prefix": "Window: "}, pad={"t": 45}, steps=steps)], hovermode="x unified", xaxis_title="Timestamp", yaxis_title="Price", height=520)
    fig.update_xaxes(rangeslider=dict(visible=True))
    fig.write_html(str(out_path), include_plotlyjs="cdn", full_html=True)
    print(f"Saved {split_name} slider HTML: {out_path}")

plots_dir = PROJECT_ROOT / "Prediction" / "plot_exports"
plots_dir.mkdir(parents=True, exist_ok=True)
build_window_slider_plot("Train", train_true, train_pred, train_time, plots_dir / "train_slider_all_windows_cv.html")
build_window_slider_plot("Validation", val_true, val_pred, val_time, plots_dir / "validation_slider_all_windows_cv.html")
build_window_slider_plot("Test", test_true, test_pred, test_time, plots_dir / "test_slider_all_windows_cv.html")

Saved Train slider HTML: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\Prediction\plot_exports\train_slider_all_windows_cv.html
Saved Validation slider HTML: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\Prediction\plot_exports\validation_slider_all_windows_cv.html
Saved Test slider HTML: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\Prediction\plot_exports\test_slider_all_windows_cv.html


## Production note
For true day-ahead use, replace decoder weather values with forecast weather for the next 96 time steps.